# OsteoClassAI v2 · 05 Pérdida ordinal y mayor resolución

Dos corridas que se comparan con la base (`convnext_tiny`, QWK 0,851):

- **A:** pérdida ordinal SORD + TTA (volteo horizontal), 224 px · ~12 min en T4
- **B:** lo mismo a 288 px · ~20 min

Además de `argmax`, cada corrida reporta el grado esperado con puntos de corte ajustados **solo en validación**.

*Entorno de ejecución → Cambiar tipo → GPU (T4)*, y luego *Ejecutar todas*.


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm kagglehub

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
!python -m osteoclass.data --root "$root" --out outputs/data > data.log && tail -n 3 data.log

## A · SORD + TTA, 224 px

In [ ]:
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model convnext_tiny --loss sord --tta --epochs 30 --patience 8 --out outputs/runs/convnext_tiny_sord
!mkdir -p /content/drive/MyDrive/OsteoClassAI/runs && cp -r outputs/runs/convnext_tiny_sord /content/drive/MyDrive/OsteoClassAI/runs/

## B · SORD + TTA, 288 px

In [ ]:
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model convnext_tiny --loss sord --tta --size 288 --epochs 30 --patience 8 --out outputs/runs/convnext_tiny_sord_288
!cp -r outputs/runs/convnext_tiny_sord_288 /content/drive/MyDrive/OsteoClassAI/runs/